# Czysty notatnik do generowania działającego (miejmy nadzieję) modelu detekcji zagrożeń na podstawie logów Wazuh
## ✨Organicznie uprawiany kod✨
Zalecam korzystanie z menedżera środowisk python `uv`, bo jest lepszy i nie tak ociężały jak `conda` czy domyślny `venv`.
Tworzenie środowiska
```
uv venv .venv --python cp3.14 # argument --python opcjonalnie cp3.14 po prostu jest szybki ;)
```
Aktywacja środowiska
```
source .venv/bin/activate     # to pewnie będzie inaczej na Windows
```
Instalacja bibliotek
```
uv pip install ipykernel pandas numpy
```

In [1]:
import re
# czy wykonywać kod upraszczający analizę danych
TESTY = False
DATA_DIR = "..//data//ait_ads"
TOKEN_RX = re.compile(r"[A-Za-z0-9+/=%_.-]{8,}")
REQ_RX = re.compile(r'"([A-Z]+)\s+(\S+)\s+HTTP/[0-9.]"\s+(\d{3})')
URL_FLAGS = {
    "urlflag_wpadmin": "wp-admin",
    "urlflag_wpcontent": "wp-content",
    "urlflag_wpjson": "wp-json",
    "urlflag_wplogin": "wp-login",
    "urlflag_xmlrpc": "xmlrpc",
    "urlflag_wpconfig": "wp-config",
    "urlflag_phpmyadmin": "phpmyadmin",
    "urlflag_backup": "backup",
    "urlflag_dump": "dump",
    "urlflag_config": "config",
    "urlflag_git": ".git",
    "urlflag_sql": "sql",
    "urlflag_readme": "readme",
    "urlflag_license": "license",
    "urlflag_test": "test",
    "urlflag_dev": "dev",
    "urlflag_secret": "secret",
    "urlflag_admin": "admin",
    "urlflag_pass": "pass",
    "urlflag_user": "user",
    "urlflag_include": "include",
    "urlflag_proc": "proc",
    "urlflag_ocs": "ocs/",
    "urlflag_remote": "remote",
    "urlflag_shell": "shell",
    "urlflag_cmd": "cmd",
    "urlflag_php": ".php",
    "urlflag_txt": ".txt",
    "urlflag_zip": ".zip",
}
UA_PATTERNS = {
    "ua_dirb": "dirb",
    "ua_wpscan": "wpscan",
    "ua_nmap": "nmap",
    "ua_ncat": "ncat",
    "ua_wget": "wget",
    "ua_headless": "HeadlessChrome",
}
SCENARIOS = ["fox", "harrison", "russellmitchell", "santos", "shaw", "wardbeck", "wheeler", "wilson"]


In [2]:
import json
import pandas as pd

def load_scenario(scenario):
    records = []
    with open(f"../data/ait_ads/{scenario}_wazuh.json", "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:  # skip blank lines
                record = json.loads(line)
                if not "Suricata" in record["rule"]["description"] and not "suricata" in record["rule"]["groups"]:
                    records.append(record)
    df = pd.json_normalize(records)
    records.clear()
    return df

df = load_scenario(SCENARIOS[0])

Debugowanie

In [3]:
def print_unique_values(df):
    for col in df.columns:
        try:
            counts = df[col].value_counts(dropna=False)
        except TypeError:  # lists or dicts aren't hashable
            counts = df[col].astype(str).value_counts(dropna=False)

        print(f"\n=== {col} ({counts.shape[0]} unique) ===")
        print(counts)

if TESTY:
    print_unique_values(df)

Wstępne oczyszczanie danych

**Your feature set has a likely leakage problem: several of the `rule.*` fields are derived from the same rules that produced the label, so the model may learn the rules rather than the attacks.** Before training, check how `attack` was labeled. If it came from Wazuh rule levels, groups, or MITRE mappings, those features will make the task trivially easy and the model won't generalize to new traffic.

Here's how I'd group your features:

| Group | Features | Recommendation |
|---|---|---|
| **Likely leakage** | `rule.level`, `rule.groups`, `rule.mitre.*`, `rule.pci_dss`, `rule.tsc`, `rule.nist_800_53`, `rule.gpg13`, `rule.gdpr`, `rule.hipaa` | Drop these if the label depends on Wazuh's rules. The compliance fields are just rule metadata, so they mostly identify the rule that fired. |
| **Identifiers** | `id`, `data.id`, `full_log` (as raw text) | Drop `id` since it's unique per alert and carries no signal. `data.id` is probably a user or session identifier and risks memorization. `full_log` needs parsing or vectorization, not direct use. |
| **Time** | `@timestamp` | Don't feed raw timestamps. Extract hour of day, day of week, and possibly inter-event gaps per source IP. |
| **Network** | `data.srcip`, `data.dstip`, `data.srcport`, `data.protocol`, `data.url` | Keep these but encode carefully. IPs are high cardinality, so use derived features like counts per IP over a time window, whether the IP is internal or external, or port categories. |
| **Identity and session** | `data.srcuser`, `data.dstuser`, `data.tty` | Usually sparse. Consider flags such as "login as root" or "failed attempts per user" instead of raw values. |
| **Geo** | `GeoLocation.*` | Mostly populated only for external IPs. Consider a missing-value indicator, and beware that country-level features can encode the data source more than attack behavior. |
| **Source metadata** | `location`, `predecoder.*`, `decoder.name`, `decoder.parent` | These describe which log produced the event. They're useful for context but can cause the model to learn log-source imbalances. |
| **Unclear** | `previous_output` | I can't tell what this is from the name. If it's the output of an earlier pipeline stage or a prior model, check that it isn't built from the label. |

Two other things to check. First, time-based splits matter more than random splits here, since attack patterns drift and random splits can inflate your scores. Second, sparse fields like `data.url` and the `GeoLocation` fields may be mostly null, so confirm their null rates before relying on them.

What does the `attack` label come from, and what's `previous_output`? With those answers I can say more precisely which features to keep.

In [4]:
df = df.drop(columns=[
    "agent.name", "agent.id", # id, agent.ip drop później
    "manager.name",
    "rule.firedtimes", "rule.mail", "rule.description", "rule.id", "rule.frequency", "rule.level", "rule.groups", "rule.mitre.technique", "rule.mitre.id", "rule.mitre.tactic", "rule.pci_dss", "rule.tsc", "rule.nist_800_53", "rule.gpg13", "rule.gdpr", "rule.hipaa", # zastąpić przez fired times ale w oknie
    "predecoder.timestamp",
    "input.type",
    "data.audit.uid", "data.audit.auid", "data.audit.gid", "data.audit.exe", "data.audit.euid", "data.audit.session", "data.audit.pid", "data.audit.id", "data.audit.type", "data.audit.command", "data.uid", "data.pwd", "data.command", "data.euid", "data.id", # wyglądają bezurzytecznie
    "decoder.ftscomment", "decoder.name", "decoder.parent",
    "GeoLocation.city_name", "GeoLocation.country_name", "GeoLocation.location.lat", "GeoLocation.location.lon", "GeoLocation.region_name", "location",
    "predecoder.hostname", 'predecoder.program_name',
    "previous_output"
], errors="ignore")

# jeżeli location = "/var/log/apache2/intranet-access.log" to host_role = WWW
# data.srcuser zamienić na is_known_user


In [5]:
if TESTY:
    print_unique_values(df)
    print(df.shape)
    print(df.columns)

Funkcje

In [6]:
from datetime import UTC, datetime
from collections import Counter
import math

def port_category(port):
    if port == 80:
        return "http"
    elif port == 53:
        return "dns"
    elif port == 443:
        return "https"
    elif port == 587:
        return "smtp"
    elif port == 5800:
        return "vnc"
    else:
        return "other"

local_ips = df['agent.ip'].value_counts(dropna=False).keys()[1:]

def is_ip_internal(ip):
    if type(ip) != str:
        return False
    split = ip.split(':')
    #local_ips = df['agent.ip'].value_counts(dropna=False).keys()[1:]
    if split[0] in local_ips:
        return True
    else:
        return False

def parse_iso(value: str) -> float:
    parsed = datetime.strptime(value, "%Y-%m-%dT%H:%M:%S.%f%z")
    return parsed.timestamp()

def hour(ts: str) -> float:
    iso_ts = parse_iso(ts)
    stamp = datetime.fromtimestamp(iso_ts, tz=UTC)
    # interval = ts # nwm to trzeba zrobić
    return float(stamp.hour)

def day(ts: str) -> float:
    iso_ts = parse_iso(ts)
    stamp = datetime.fromtimestamp(iso_ts, tz=UTC)
    # interval = ts # nwm to trzeba zrobić
    return float(stamp.weekday())

def entropy_of(text: object) -> float:
    if type(text) != str:
        return 0.0
    counts = Counter(text)
    total = len(text)
    value = 0.0
    for count in counts.values():
        probability = count / total
        value -= probability * math.log2(probability)
    return round(value, 4)

Generowanie istotnych cech

In [7]:
import numpy as np
df["unix_time"] = df["@timestamp"].apply(lambda x: parse_iso(x))

df["is_dst_user_root"] = np.where(df["data.dstuser"] == "root", 1, 0)
# data.srcip -> ip wew lub zew
df["is_srcip_internal"] = df["data.srcip"].apply(is_ip_internal)
# data.dstip -> ip wew lub zew
df["is_dstip_internal"] = df["data.dstip"].apply(is_ip_internal)
# full_log -> wektoryzacja

# @timestamp -> godzina_dnia, dzień_tygodnia, czas_między_ostatnim_tym_samym_srcip
df["hour"] = df["@timestamp"].apply(hour)
df["weekday"] = df["@timestamp"].apply(day)
# kategoria portu
df["dstport"] = df["data.dstip"].apply(port_category)
# cechy url
df["url_entropy"] = df["data.url"].apply(entropy_of)
df["url_len"] = df["data.url"].apply(lambda x: len(x) if type(x) == str else 0 )
df["url_parts"] = df["data.url"].apply(lambda x: len(x.split("/")) if type(x) is str else 0)
df["url_query"] = df["data.url"].apply(lambda x: "?" in x if type(x) is str else 0)
## wymagane do needle scanu, po wszystkim url i tak trzeba usunąć
df["data.url"] = df["data.url"].apply(lambda x: x.lower() if type(x) is str else 0)
## needle scan
for flag, needle in URL_FLAGS.items():
    df[flag] = df["data.url"].apply(lambda x: float(needle in x if type(x) is str else False))
for flag, needle in UA_PATTERNS.items():
    df[flag] = df["data.url"].apply(lambda x: float(needle.lower() in x if type(x) is str else False))
df["url_tokens"] = df["data.url"].apply(lambda x: float(len(re.findall(r"[A-Za-z0-9]+", x)) if type(x) is str else 0))
df["url_encoded_chars"] = df["data.url"].apply(lambda x: float(x.count("%") if type(x) is str else 0))
df["url_max_token"] = df["data.url"].apply(lambda x: float(max((len(t) for t in TOKEN_RX.findall(x)), default=0)) if type(x) is str else 0)
# ograniczenie full_log
df["log_head"] = df["full_log"].apply(lambda x: re.sub(r'\b\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}\b', '', x[:400]) if type(x) is str else "")
# oznaczenie scenariusza
df["scenario"] = "fox"

Oznaczanie okien ataków

In [8]:
import csv

def load_phases(path: str) -> pd.DataFrame:
    rows = []
    with open(path, newline="", encoding="utf-8") as handle:
        for row in csv.DictReader(handle):
            rows.append({
                'scenario':    row["scenario"].strip(),
                'attack_type': row["attack"].strip(),
                'start':       float(row["start"].strip()),
                'end':         float(row["end"].strip()),
            })
    return pd.DataFrame(rows, columns=['scenario', 'attack_type', 'start', 'end'])

def classify_vectorized(df, phases):
    labels = np.full(len(df), "background", dtype=object)
    types  = np.full(len(df), "", dtype=object)
    for _, phase in phases.iterrows():
        mask = (df["unix_time"] >= phase["start"]) & (df["unix_time"] < phase["end"])
        labels[mask] = "attack"
        types[mask]  = phase["attack_type"]
    return pd.DataFrame({"attack": labels, "attack_type": types}, index=df.index)

phases = load_phases("../data/ait_ads/labels.csv")
df = pd.concat([df, classify_vectorized(df, phases)], axis=1)

Wtórne oczyszczanie danych

In [9]:
df = df.drop(columns=["full_log", "agent.ip", "data.url", "data.srcip", "data.dstip", "data.dstuser", "data.srcuser"], errors="ignore")

In [10]:
# może się przydać, ale nie teraz
def scan_text_for_url(raw: str) -> tuple[str, str, str]:
    match = REQ_RX.search(raw)
    if match:
        return match.group(2), match.group(1), match.group(3)
    return "", "", ""

# może się przydać, ale nie teraz
def log_resource_class(resources: list) -> str:
    text = " ".join(str(r) for r in resources).lower()
    if "access" in text or "apache" in text:
        return "apache"
    if "metricbeat" in text or "filebeat" in text:
        return "metrics"
    if "syslog" in text or "auth" in text:
        return "syslog"
    return "other"

In [11]:
# firedtimes wewnątrz okna

## Wytyczanie okien czasowych, ekstrakcja danych z okna oraz enkodowanie cech

W tym kroku przekształcamy surowe zdarzenia w bezpieczny wektor cech bez wycieku adresów IP, znaczników czasu oraz identyfikatorów reguł:
1. **Wytyczanie okien**: sortujemy logi chronologicznie, wyznaczamy odstęp od poprzedniego zdarzenia (`time_diff_prev`), czas od początku bieżącego okna aktywności (`time_since_window_start`, reset po przerwie > 60 s) oraz okna przesuwne **60 s** i **600 s**.
2. **Dane z okna**:
   - łączna liczba zdarzeń w oknie (`events_in_60s`, `events_in_600s`),
   - lokalna częstotliwość reguły w oknie 60 s (`rule_fired_in_60s` zamiast narastającego `rule.firedtimes` i bez podawania `rule.id`),
   - liczba unikalnych źródeł w oknie (`unique_srcip_in_60s`) oraz aktywność z tego samego źródła (`same_srcip_in_60s`, `srcip_time_diff`),
   - rola hosta (`host_role`: `WWW`, `mail`, `dns`, `auth`, `other`) wyznaczona z `location` oraz klasa sieci (`srcip_net_class`, `dstip_net_class`: sieć wewnętrzna/zewnętrzna) zamiast surowych IP.
3. **Enkodowanie cech**:
   - maskowanie adresów IP, domen (np. `.biz`), nazw plików i dat w `full_log` oraz sprowadzenie tekstu do małych liter (`lowercase=True` — równoważność wszystkich liter) przed wektoryzacją TF-IDF,
   - kodowanie cech kategorycznych (`OrdinalEncoder`) i usunięcie wszystkich surowych kolumn źródłowych.

Komórka 2 wytyczanie okien i obliczanie cech z okna

In [12]:
import ipaddress
from collections import Counter
import numpy as np
import pandas as pd

def extract_host_role(loc: object) -> str:
    if not isinstance(loc, str):
        return "other"
    l = loc.lower()
    if any(k in l for k in ["apache", "nginx", "access.log", "error.log", "www", "http"]):
        return "WWW"
    if any(k in l for k in ["mail", "postfix", "dovecot", "exim", "smtp", "imap"]):
        return "mail"
    if any(k in l for k in ["bind", "named", "dns", "unbound"]):
        return "dns"
    if any(k in l for k in ["auth", "secure", "sshd", "audit"]):
        return "auth"
    return "other"

def check_private_ip(ip_val: object) -> int:
    if not isinstance(ip_val, str) or not ip_val.strip():
        return -1  # brak adresu IP
    clean_ip = ip_val.split(":")[0].strip()
    try:
        return int(ipaddress.ip_address(clean_ip).is_private)
    except ValueError:
        return -1

def rolling_count_by_group(df_sorted: pd.DataFrame, group_series: pd.Series, window: float = 60.0) -> np.ndarray:
    times = df_sorted["unix_time"].to_numpy()
    result = np.zeros(len(df_sorted), dtype=np.int32)
    groups_dict = df_sorted.groupby(group_series).indices
    for idx in groups_dict.values():
        t = times[idx]
        result[idx] = np.arange(len(t)) - np.searchsorted(t, t - window, side="left")
    return result

# 1. Sortowanie chronologiczne (w ramach scenariusza, jeśli kolumna istnieje)
sort_cols = ["scenario", "unix_time"] if "scenario" in df.columns else ["unix_time"]
df = df.sort_values(sort_cols).reset_index(drop=True)

# 2. Wytyczanie okien czasowych (odstępy i okna przesuwne 60s / 600s)
times = df["unix_time"].to_numpy()
df["time_diff_prev"] = df["unix_time"].diff().fillna(0.0)
if "scenario" in df.columns:
    df.loc[df["scenario"] != df["scenario"].shift(1), "time_diff_prev"] = 0.0

# Czas od początku bieżącego okna aktywności (nowe okno po przerwie > 60s)
window_id = (df["time_diff_prev"] > 60.0).cumsum()
df["time_since_window_start"] = df["unix_time"] - df.groupby(window_id)["unix_time"].transform("first")

# Liczba wszystkich alertów w oknach 60s i 600s
df["events_in_60s"] = np.arange(len(times)) - np.searchsorted(times, times - 60.0, side="left")
df["events_in_600s"] = np.arange(len(times)) - np.searchsorted(times, times - 600.0, side="left")

# 3. Dane z okna: zastąpienie rule.firedtimes licznikiem odpaleń tej samej reguły w oknie 60s
if "rule.id" in df.columns:
    df["rule_fired_in_60s"] = rolling_count_by_group(df, df["rule.id"].astype(str), window=60.0)
else:
    df["rule_fired_in_60s"] = 0

# 4. Dane z okna: rola hosta (zamiast location) oraz anonimizacja IP źródłowego i docelowego
df["host_role"] = df["location"].apply(extract_host_role) if "location" in df.columns else "other"

src_series = df["data.srcip"].fillna("").astype(str) if "data.srcip" in df.columns else pd.Series("", index=df.index)
dst_series = df["data.dstip"].fillna("").astype(str) if "data.dstip" in df.columns else pd.Series("", index=df.index)

df["srcip_net_class"] = src_series.apply(check_private_ip)
df["dstip_net_class"] = dst_series.apply(check_private_ip)

# Liczba unikalnych adresów źródłowych (srcip) w oknie 60s (algorytm dwuwskaźnikowy O(N))
src_vals = src_series.to_numpy()
unique_srcs = np.empty(len(times), dtype=np.int32)
active_counts = Counter()
left = 0
for right in range(len(times)):
    s_in = src_vals[right]
    if s_in != "":
        active_counts[s_in] += 1
    while times[right] - times[left] > 60.0:
        s_out = src_vals[left]
        if s_out != "":
            active_counts[s_out] -= 1
            if active_counts[s_out] == 0:
                del active_counts[s_out]
        left += 1
    unique_srcs[right] = len(active_counts)
df["unique_srcip_in_60s"] = unique_srcs

# Aktywność z tego samego źródła w oknie 60s i odstęp czasu od poprzedniego logu z tego źródła
src_group = pd.Series(np.where(src_series == "", "no_srcip", src_series), index=df.index)
df["same_srcip_in_60s"] = np.where(src_series == "", 0, rolling_count_by_group(df, src_group, window=60.0))
df["srcip_time_diff"] = np.where(src_series == "", -1.0, df.groupby(src_group)["unix_time"].diff().fillna(9999.0))

# Poprawne wyznaczenie kategorii portu docelowego z data.dstport
if "data.dstport" in df.columns:
    df["dstport_cat"] = pd.to_numeric(df["data.dstport"], errors="coerce").apply(port_category)
else:
    df["dstport_cat"] = "other"

Komórka 3 maskowanie tekstu i enkodowanie wartości cech

In [13]:
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OrdinalEncoder

def mask_log_tokens(text: object) -> str:
    if not isinstance(text, str):
        return ""
    s = text[:400].lower()  # wszystkie litery są sobie równoważne
    # Maskowanie adresów IPv4 oraz IPv6
    s = re.sub(r"\b\d{1,3}(?:\.\d{1,3}){3}\b", " ip_token ", s)
    s = re.sub(r"\b[0-9a-f]{1,4}(?::[0-9a-f]{1,4}){2,7}\b", " ipv6_token ", s)
    # Maskowanie domen i TLD (np. domeny DNSteal z .biz, .com, .local itp.)
    s = re.sub(r"\b(?:[a-z0-9-]+\.)+(?:biz|com|net|org|io|info|local|lan|intranet)\b", " domain_token ", s)
    # Maskowanie nazw plików z rozszerzeniami
    s = re.sub(r"\b[a-z0-9_.-]+\.(?:php|txt|zip|sql|sh|py|exe|bak|tar|gz|log)\b", " file_token ", s)
    # Maskowanie znaczników czasu
    s = re.sub(r"\b\d{4}-\d{2}-\d{2}[t\s]\d{2}:\d{2}:\d{2}(?:\.\d+)?(?:z|[+-]\d{2}:?\d{2})?\b", " ts_token ", s)
    return s

# 1. Enkodowanie cech kategorycznych (host_role, dstport_cat)
cat_cols = ["host_role", "dstport_cat"]
cat_encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
encoded_cats = cat_encoder.fit_transform(df[cat_cols].astype(str))
df_cats = pd.DataFrame(encoded_cats, columns=["host_role_enc", "dstport_enc"], index=df.index)

# 2. Enkodowanie danych tekstowych z zamaskowanego logu (lowercase=True, bez ostrzeżeń Pylance)
log_source_col = "full_log" if "full_log" in df.columns else "log_head"
masked_logs = df[log_source_col].apply(mask_log_tokens)

tfidf = TfidfVectorizer(
    lowercase=True,
    max_features=35,
    token_pattern=r"[a-z_]{3,}"
)
tfidf_sparse = tfidf.fit_transform(masked_logs)
tfidf_cols = [f"tfidf_{w}" for w in tfidf.get_feature_names_out()]
df_tfidf = pd.DataFrame.sparse.from_spmatrix(tfidf_sparse, columns=tfidf_cols, index=df.index).sparse.to_dense()

# 3. Złożenie finalnej macierzy zakodowanych cech numerycznych X (bez dat, godzin, surowych IP i ID reguł)
feature_cols = [
    "time_diff_prev", "time_since_window_start", "events_in_60s", "events_in_600s",
    "rule_fired_in_60s", "srcip_net_class", "dstip_net_class",
    "unique_srcip_in_60s", "same_srcip_in_60s", "srcip_time_diff",
    "is_dst_user_root", "url_entropy", "url_len", "url_parts", "url_query",
    "url_tokens", "url_encoded_chars", "url_max_token"
] + list(URL_FLAGS.keys()) + list(UA_PATTERNS.keys())

existing_num_cols = [c for c in feature_cols if c in df.columns]
X = pd.concat([
    df[existing_num_cols].astype(float).fillna(0.0),
    df_cats,
    df_tfidf
], axis=1)

print("Zakodowana macierz cech X:", X.shape)

Zakodowana macierz cech X: (451439, 90)


(Uwaga: aby Komórka 2 mogła wyliczyć rule_fired_in_60s, host_role, dstport_cat oraz statystyki srcip, uruchom te komórki przed starymi komórkami usuwającymi kolumny rule.id, location, data.dstport, data.srcip i full_log — wynikowa tabela X na końcu Komórki 3 sama odrzuca wszystkie surowe pola).